# Limpieza y transformación de datos
En este cuaderno se trabajará con los datos ingeridos correspondientes al historial de mensajes de un grupo de Whatsapp. Dichos datos fueron importados en el notebook 1.0-vd-ingesta_de_datos.ipnyb. La finalidad de este cuaderno es procesar los datos en formato tidy para después realizar un análisis exploratorio de datos (EDA).

In [25]:
# Importacion de librerias
import pandas as pd
import os

# Definir la ruta donde se encuentra el archivo 
ruta = os.path.join('..', 'data', 'interim', 'historial_whatsapp_interim.csv')

# Leer archivo df_raw.csv
df_raw = pd.read_csv(ruta, index_col = 0 )

# Vamos a ver el tipo de dato  que se tiene en cada columna (se espera un string)
print(df_raw.dtypes)

# Visualizar el df
df_raw.head(1) # para mantener la privacidad de los datos, no se mostrará a detalle el contenido del archivo. Para fines prácticos, solamente se mostrará uno de los mensajes que fueron enviados por mi (Victor)


timestamp_raw       str
usuario_original    str
contenido           str
dtype: object


,timestamp_raw,usuario_original,contenido
0,"24/5/2024, 11:27 a. m.",Victor,Cambié de cel y se me borraron todos los datos...


__Para mantener la privacidad de los datos, no se mostrará a detalle el contenido del archivo. Para fines prácticos, solamente se mostrará uno de los mensajes que fueron enviados por mi (Victor)__

Primeramente, se trabajará con la columna **timestamp_raw**. Para poder convertir estos datos a formato datetime, necesitamos corregir el "a.m." y "p.m." que se encuentra al final del df. . Después, podemos utilizar el método pd.to_datetime() para darle un formato especifico a la columna. 

In [26]:
# copiar df para trabajar de manera segura
df = df_raw.copy()

# Estandarizar 'a.m.' y 'p.'m' a 'AM y 'PM'
df["timestamp_clean"] = (
    df["timestamp_raw"]
    .str.replace(" a. m.", " AM", regex=False) # utilizamos regex false para buscar literalmente en la cadena y no como expresion regular
    .str.replace(" p. m.", " PM", regex=False))

# Aplicar el metodo de pandas to_datetime utilizando el siguiente formato: dia-mes-año, hora-minuto-am/pm (%d/%m/%Y, %I:%M %p)
df["timestamp"] = pd.to_datetime(df["timestamp_clean"], format="%d/%m/%Y, %I:%M %p")

# Eliminar las columnas de timestamp crudas y limpitas
df.drop(columns=['timestamp_raw','timestamp_clean'], inplace=True)

# Re ordenar por columnas
df = df[['timestamp', 'usuario_original', 'contenido']]
df.head(1)

,timestamp,usuario_original,contenido
0,2024-05-24 11:27:00,Victor,Cambié de cel y se me borraron todos los datos...


Ahora se trabajará con la columna usuario_original, en la cual se cambiaran los nombres de las personas por personajes de las peliculas de Shrek. 

In [ ]:
# copiar df para trabajar de manera segura
df = df.copy()

# Para obtener los nombres de cada usuario se puede usar el metodo .unique(). 
print(df['usuario_original'].unique()) #para mantener la privacidad de los datos, no se mostrará a detalle el contenido del archivo. Para fines prácticos, solamente se mostrará uno de los mensajes que fueron enviados por mi (Victor)

# Diccionario para mapear los nombres reales con los personajes de shrek
# Para esta parte, se han eliminado los nombres originales de los usuarios que vienen en el archivo original
# Para correr este script sin error, hay que sustituir donde diga 'usuario_xi' por el nombre del usuario 
#personajes_shrek = {'usuario_1': 'Shrek, 'usuario_2':'Lord Farquad', 'usuario_3':'Burro', 'usuario_4':'Fiona', 'usuario_5':'Lobo de Sexo Dudoso'}

# Diccionario para mapear los nombres reales con los personajes de shrek
# Para esta parte, se han eliminado los nombres originales de los usuarios que vienen en el archivo original
# Para correr este script sin error, hay que sustituir donde diga 'usuario_xi' por el nombre del usuario 
personajes_shrek = {'usuario_1': 'Shrek', 'usuario_2':'Lord Farquad', 'usuario_3':'Burro', 'usuario_4':'Fiona', 'usuario_5':'Lobo de Sexo Dudoso'}

# Mapear los nombres reales con los de personajes de Shrek
df['personaje'] = df['usuario_original'].map(personajes_shrek)

# Confirmar los nombres en la nueva columna de personaje
print('\n',df['personaje'].unique())

# Eliminar la columna con los nombres originales de los usuarios
df.drop(columns='usuario_original', inplace=True)

# Re ordenar df
df = df[['timestamp','personaje', 'contenido']]
df.head()


<StringArray>
['Victor', 'Efrén Camargo', 'Pablo', 'Carlos', 'Meta AI']
Length: 5, dtype: str

 <StringArray>
['Shrek', 'Lord Farquad', 'Fiona', 'Burro', 'Lobo de Sexo Dudoso']
Length: 5, dtype: str


,timestamp,personaje,contenido
0,2024-05-24 11:27:00,Shrek,Cambié de cel y se me borraron todos los datos...
1,2024-05-24 11:27:00,Shrek,‎STK-20240524-WA0002.webp (archivo adjunto)
2,2024-05-24 11:31:00,Lord Farquad,We que chilo alv
3,2024-05-24 11:31:00,Shrek,Y si
4,2024-05-24 11:31:00,Lord Farquad,La quisieras


Para trabajar con la columna de conternido, tenemos que primero clasificar los tipos de contenido que puden existir en un mensaje de whatsapp:
*  __texto__ = cualquier mensaje que hay en una conversacion normal y que no sea un archivo ni un elemento multimedia
* __video__: contiene extensiones de video (.mp4, .mkv, .mov) o el texto video omitido.
* __imagen__: contiene extensiones de imagen (.jpg, .jpeg, .png) o el texto imagen omitida.
* __audio__: contiene extensiones de audio de WhatsApp (.opus, .mp3, .m4a, .aac, audio omitido)
* __archivo__: contiene documentos o ejecutables (.pdf, .docx, .xlsx, .zip, etc.) o la leyenda genérica (archivo adjunto) que no haya entrado en las categorías anteriores
* __sticker__ : contiene .webp o la leyenda STK-.


In [28]:
# Importacion del mudlo re para trabajar con expresiones regulares
import re

# Creacion de funcion para clasificar los contenidos de los mensajes
def clasificar_contenido(mensaje):
    if not isinstance(mensaje, str): # Si obtenemos un valor faltante, u otro valor que no sea str, lo asignaremos como texto
        return "texto"

    texto = mensaje.lower().strip() # cortar espacios en blanco y hacer las palabras minuscula

    #  Stickers
    if ".webp" in texto or "stk-" in texto:
        return "sticker"

    # 2. Videos
    elif any(formato in texto for formato in [".mp4", ".mkv", ".mov", "video omitido"]):
        return "video"

    # 3. Imágenes
    elif any(formato in texto for formato in [".jpg", ".jpeg", ".png", "imagen omitida", "foto omitida"]):
        return "imagen"

    # 4. Audios 
    elif any(formato in texto for formato in [".opus", ".mp3", ".m4a", ".aac", "audio omitido"]):
        return "audio"

    # 5. Otros Archivos (PDFs, docs, zips o adjuntos genéricos)
    elif (any(formato in texto for formato in [".pdf",".docx",".xlsx",".pptx",".zip",".rar",".csv",".txt",]) or "(archivo adjunto)" in texto):
        return "archivo"

    # 6. Texto plano ordinario
    else:
        return "texto"


In [29]:
# Aplicar la funcion para clasificar el contenido y agregar una nueva columna
df["tipo_contenido"] = df["contenido"].apply(clasificar_contenido)

# Confirmar los conteos de cada categoría
print("Distribución de tipos de contenido:")
print(df["tipo_contenido"].value_counts())

Distribución de tipos de contenido:
tipo_contenido
texto      81472
sticker     1933
audio        970
imagen       182
video         26
archivo        8
Name: count, dtype: int64


In [30]:
# Ordenar las columnas del df
# Conservar solo columnas necesarias en orden lógico
df_procesado = df[["timestamp", "personaje", "tipo_contenido", "contenido"]]

# Vista previa final
df_procesado.tail()

,timestamp,personaje,tipo_contenido,contenido
84586,2026-09-21 11:48:00,Lord Farquad,texto,Del primer dia
84587,2026-09-21 11:48:00,Lord Farquad,texto,Si
84588,2026-09-21 11:48:00,Lord Farquad,texto,Ese dia full explosion mental
84589,2026-09-21 11:48:00,Lord Farquad,texto,Me recordo a la primera probada de mota
84590,2026-09-21 11:58:00,Burro,texto,Alv jajajajajaja bien intenso


Finalmente, para mantener el anonimato de los mensajes se sustituirán los nombres reales que aparecen dentro del texto por el nombre de los personajes de shrek. Para esto, tomaremos en cuenta las menciones que llevan el simbolo "@", asi como el nombre de las personas. 

In [31]:
# Diccionario para mapear los nombres reales con los personajes de shrek
personajes_shrek = {'Victor': 'Shrek', 'Efrén Camargo':'Lord Farquad', 'Carlos':'Burro', 'Pablo':'Fiona', 'Meta AI':'Lobo de Sexo Dudoso'}

# Reemplazar los nombres mencionados con "@" por los nombres de los personajes de shrek
for nombre_real, personaje in personajes_shrek.items():
    df['contenido'] = df['contenido'].str.replace(f"@{nombre_real}", f"@{personaje}", regex=False)

# Reemplazar las menciones que no tienen arroba
for nombre_real, personaje in personajes_shrek.items():
    df['contenido'] = df['contenido'].str.replace(nombre_real, personaje, regex=False)



# Confirmar que los datos ya no contienen nombres personales en los mensajes
# Filtro para obtener strings que tengan el arroba
filtro = df["contenido"].str.contains("@")
# Aplicar filtro a las columnas de personaje y contenido
df[filtro][["personaje", "contenido"]].head(10)


,personaje,contenido
266,Fiona,@⁨Burro⁩
289,Lord Farquad,Oye @⁨Shrek⁩
443,Burro,@⁨Fiona⁩ cuando son las finales de la nba
534,Lord Farquad,@⁨Shrek⁩ que haras en tu cumple
614,Lord Farquad,@⁨Burro⁩
640,Lord Farquad,@⁨Burro⁩ de casualidad viniste a la casa?
840,Lord Farquad,En que parte esta @⁨Shrek⁩
883,Shrek,que costco es o que jajaja @⁨Lord Farquad⁩
902,Lord Farquad,@⁨Shrek⁩ puedo acampar en tu casa?
913,Fiona,@⁨Burro⁩ RAÚL


In [32]:
# Exportacion del dataframe a un archivo .csv
import os

# Definicion de la ruta en donde se guardaran los datos
ruta = os.path.join ('..', 'data', 'processed', 'datos_procesados.csv')

# Exportar archivo csv
df.to_csv(ruta, encoding="utf-8-sig")